<a href="https://colab.research.google.com/github/beingAnujChaudhary/Practical-Statistics-for-DS/blob/main/chapter_04_regression_and_prediction/experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/Practical-Statistics-for-DS.git

# Move into repository
%cd /content/Practical-Statistics-for-DS

# Move into Chapter 4 folder
%cd chapter_04_regression_and_prediction

Mounted at /content/drive
Cloning into 'Practical-Statistics-for-DS'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 180 (delta 109), reused 70 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (180/180), 604.30 KiB | 3.80 MiB/s, done.
Resolving deltas: 100% (109/109), done.
/content/Practical-Statistics-for-DS
/content/Practical-Statistics-for-DS/chapter_04_regression_and_prediction


# Chapter 04 Experiments: Regression and Prediction

> Source: *Practical Statistics for Data Scientists, 2nd Edition*  
> Chapter 4: Regression and Prediction

## Purpose

This notebook is for:
- Experimentation and intuition building
- Understanding prediction quality
- Testing regression assumptions
- Exploring model failures and diagnostics
- Building intuition for when and why models break

Unlike `exercises.ipynb`, there are no fixed answers.

Goal:

> "How do relationships affect prediction, and when do models fail?"

---

## 🧪 How to Use This Notebook

- **Experiments ≠ Exercises**: These are open-ended explorations designed to build your intuition for regression diagnostics, model failures, and the transition from explanatory statistics to predictive machine learning.
- **Modify freely**: Change the noise levels, outlier magnitudes, and polynomial degrees to see how the models break.
- **Document insights**: Add your observations in markdown cells as you go.
- **Save interesting outputs**: Export plots to `output/` for your portfolio.

---

## Setup

<details>
<summary>Click to view solution</summary>

```python
# Imports
try:
    from utils.notebook_setup import *
except:
    import warnings
    warnings.filterwarnings("ignore")
    
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    
    plt.style.use("seaborn-v0_8")
    np.random.seed(42)

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression, HuberRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

# Create output directory
os.makedirs('output', exist_ok=True)

print("Experiment notebook ready")
```
</details>


---

# Experiment 1: Strong Signal vs Weak Signal

## Question

What happens when relationship strength changes?

### Hypothesis

Stronger relationship → better predictions, higher R².

<details>
<summary>Click to view solution</summary>

```python
# Strong signal
x_strong = np.random.normal(50, 10, 300)
y_strong = x_strong * 10 + np.random.normal(0, 10, 300)

# Weak signal
x_weak = np.random.normal(50, 10, 300)
y_weak = x_weak * 10 + np.random.normal(0, 200, 300)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(x_strong, y_strong)
ax[0].set_title("Strong Signal (Low Noise)")
ax[1].scatter(x_weak, y_weak)
ax[1].set_title("Weak Signal (High Noise)")
plt.show()
```
</details>

## Reflection

Questions:
1. Which relationship looks clearer?
2. Why does noise matter?
3. Which dataset should predict better?


---

# Experiment 2: Noise vs Prediction Quality

## Question

How does noise affect prediction quality?

### Hypothesis

More noise → lower R², higher prediction error.

<details>
<summary>Click to view solution</summary>

```python
noise_levels = [10, 50, 100, 200]
results = []

for noise in noise_levels:
    X = np.random.normal(50, 10, 300)
    y = X * 10 + np.random.normal(0, noise, 300)
    
    X_reshaped = X.reshape(-1, 1)
    model = LinearRegression()
    model.fit(X_reshaped, y)
    predictions = model.predict(X_reshaped)
    
    results.append({
        "Noise": noise,
        "R²": r2_score(y, predictions),
        "MSE": mean_squared_error(y, predictions)
    })

results_df = pd.DataFrame(results)
print(results_df)

plt.figure(figsize=(8, 5))
plt.plot(results_df["Noise"], results_df["R²"], marker="o", linewidth=2)
plt.xlabel("Noise Level")
plt.ylabel("R² Score")
plt.title("Noise vs Prediction Quality")
plt.grid(alpha=0.3)
plt.show()
```
</details>

## Reflection

Questions:
1. What happened as noise increased?
2. Why did R² fall?
3. Why is real-world prediction hard?



---

# Experiment 3: Coefficient Intuition

## Question

How does slope affect coefficient estimation?

### Hypothesis

Steeper relationship → larger estimated coefficient.

<details>
<summary>Click to view solution</summary>

```python
slopes = [1, 5, 10, 20]
coefficients = []

for slope in slopes:
    x = np.random.normal(50, 10, 300)
    y = x * slope + np.random.normal(0, 20, 300)
    
    x_reshaped = x.reshape(-1, 1)
    model = LinearRegression()
    model.fit(x_reshaped, y)
    coefficients.append(model.coef_[0])

coefficient_df = pd.DataFrame({
    "True Slope": slopes,
    "Estimated Coefficient": coefficients
})
print(coefficient_df)
print("\nEstimates should be close to true slopes, with some variation due to noise.")
```
</details>

## Reflection

Questions:
1. Were coefficients accurate?
2. Why are estimates imperfect?
3. Why does noise matter?



---

# Experiment 4: Multiple Regression Behaviour

## Question

Do more features improve prediction?

### Hypothesis

Useful features improve model quality.

<details>
<summary>Click to view solution</summary>

```python
# Create dataset with multiple features
dataset = pd.DataFrame({
    "Size": np.random.normal(1500, 300, 500),
    "Bedrooms": np.random.randint(1, 6, 500),
    "Age": np.random.randint(1, 50, 500)
})

dataset["Price"] = (
    dataset["Size"] * 200 +
    dataset["Bedrooms"] * 10000 -
    dataset["Age"] * 500 +
    np.random.normal(0, 25000, 500)
)

# Simple model (Size only)
X_simple = dataset[["Size"]]
y = dataset["Price"]
simple_model = LinearRegression().fit(X_simple, y)
simple_pred = simple_model.predict(X_simple)

# Multiple model (all features)
X_multiple = dataset[["Size", "Bedrooms", "Age"]]
multiple_model = LinearRegression().fit(X_multiple, y)
multiple_pred = multiple_model.predict(X_multiple)

print(f"Simple Regression R²: {r2_score(y, simple_pred):.4f}")
print(f"Multiple Regression R²: {r2_score(y, multiple_pred):.4f}")
print(f"RMSE Simple: {np.sqrt(mean_squared_error(y, simple_pred)):.0f}")
print(f"RMSE Multiple: {np.sqrt(mean_squared_error(y, multiple_pred)):.0f}")
```
</details>

## Reflection

Questions:
1. Did multiple regression improve results?
2. Why did extra information help?
3. Can too many features become dangerous?


---

# Experiment 5: Sample Size vs Stability

## Question

Does more data create more stable coefficients?

### Hypothesis

Larger datasets → better estimates.

<details>
<summary>Click to view solution</summary>

```python
sample_sizes = [20, 50, 100, 500, 1000]
estimated_coefficients = []

for n in sample_sizes:
    x = np.random.normal(50, 10, n)
    y = x * 5 + np.random.normal(0, 20, n)
    
    x_reshaped = x.reshape(-1, 1)
    model = LinearRegression()
    model.fit(x_reshaped, y)
    estimated_coefficients.append(model.coef_[0])

stability_df = pd.DataFrame({
    "Sample Size": sample_sizes,
    "Estimated Coefficient": estimated_coefficients
})
print(stability_df)
print(f"\nTrue coefficient: 5.0")
print("Larger samples produce estimates closer to the true value.")
```
</details>

## Reflection

Questions:
1. Which sample sizes looked most stable?
2. Why does more data help?
3. Why can small samples mislead?


---

# Experiment 6: Multicollinearity

## Question

What happens when features become highly correlated?

### Hypothesis

Coefficients may become unstable and hard to interpret.

<details>
<summary>Click to view solution</summary>

```python
# Create highly correlated features
size = np.random.normal(1500, 300, 500)
rooms = size / 300 + np.random.normal(0, 0.5, 500)  # Highly correlated with size

price = size * 200 + rooms * 10000 + np.random.normal(0, 25000, 500)

multi_df = pd.DataFrame({"Size": size, "Rooms": rooms, "Price": price})

print("Correlation matrix:")
print(multi_df.corr().round(3))

plt.figure(figsize=(8, 6))
sns.heatmap(multi_df.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap (High Multicollinearity)")
plt.show()

# Fit model
X = multi_df[["Size", "Rooms"]]
y = multi_df["Price"]
model = LinearRegression().fit(X, y)

print("\nCoefficients:")
for name, coef in zip(X.columns, model.coef_):
    print(f"  {name}: {coef:.2f}")
print("Note: Coefficients may be unstable due to high correlation between Size and Rooms.")
```
</details>

## Reflection

Questions:
1. Why are coefficients unstable?
2. Why is multicollinearity dangerous?
3. Why can interpretation become difficult?


---

# Experiment 7: Multicollinearity & VIF (Variance Inflation Factor)

### Goal

Quantify multicollinearity using VIF and observe how it destroys statistical significance without necessarily hurting predictive accuracy.

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)
n = 500

# X1 and X2 are highly correlated
x1 = np.random.normal(100, 15, n)
x2 = x1 * 0.9 + np.random.normal(0, 5, n)  # Highly correlated with X1

# X3 is independent
x3 = np.random.normal(20, 5, n)

# True relationship: Y depends on X1 and X3, but NOT X2
y = 5 * x1 + 10 * x3 + np.random.normal(0, 20, n)

df = pd.DataFrame({'X1': x1, 'X2': x2, 'X3': x3, 'Y': y})

# Fit OLS Model
X_ols = sm.add_constant(df[['X1', 'X2', 'X3']])
model_ols = sm.OLS(df['Y'], X_ols).fit()

print("OLS Regression Results:")
print(model_ols.summary().tables[1])

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["Feature"] = X_ols.columns[1:]  # Exclude constant
vif_data["VIF"] = [variance_inflation_factor(X_ols.values, i) for i in range(1, X_ols.shape[1])]

print("\nVariance Inflation Factor (VIF):")
print(vif_data)
print("\nRule of thumb: VIF > 5 = moderate multicollinearity; VIF > 10 = severe.")
```
</details>

### 🤔 Reflection Prompts
1. Look at the p-value for `X2`. Is it statistically significant? Now look at its VIF.
2. If you drop `X2` from the model, how does the standard error of `X1` change?
3. **ML Relevance**: If your goal is purely prediction (not explanation), does multicollinearity actually matter? How does Ridge/Lasso handle this compared to OLS?


---

# Experiment 8: Confounding Variables

## Question

Can hidden variables create fake relationships?

### Hypothesis

Yes — confounding variables can create spurious correlations.

<details>
<summary>Click to view solution</summary>

```python
# Generate data with hidden confounder (temperature)
temperature = np.random.normal(30, 5, 500)
ice_cream_sales = temperature * 10 + np.random.normal(0, 5, 500)
swimming = temperature * 8 + np.random.normal(0, 5, 500)

confounding_df = pd.DataFrame({
    "Temperature": temperature,
    "Ice_Cream": ice_cream_sales,
    "Swimming": swimming
})

print("Correlation matrix:")
print(confounding_df.corr().round(3))
print("\nIce cream and swimming appear correlated (r > 0.9),")
print("but temperature is the hidden cause of both.")

# Pairplot
sns.pairplot(confounding_df)
plt.show()
```
</details>

## Reflection

Questions:
1. Are ice cream and swimming correlated?
2. What hidden variable exists?
3. Why does correlation mislead without considering confounders?


---

# Experiment 9: Overfitting Simulation

## Question

Can complex models learn noise instead of signal?

### Hypothesis

Very complex models overfit — they memorize training data but generalize poorly.

<details>
<summary>Click to view solution</summary>

```python
# Generate simple linear data
x = np.linspace(0, 10, 100)
y = 3 * x + np.random.normal(0, 3, 100)
x_reshaped = x.reshape(-1, 1)

# Simple and complex models
simple_model = make_pipeline(PolynomialFeatures(1), LinearRegression())
complex_model = make_pipeline(PolynomialFeatures(15), LinearRegression())

simple_model.fit(x_reshaped, y)
complex_model.fit(x_reshaped, y)

simple_predictions = simple_model.predict(x_reshaped)
complex_predictions = complex_model.predict(x_reshaped)

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(x, y, label="Data", alpha=0.6)
plt.plot(x, simple_predictions, label="Simple Model (Degree 1)", linewidth=2)
plt.plot(x, complex_predictions, label="Complex Model (Degree 15)", linewidth=2)
plt.legend()
plt.title("Overfitting Example: Simple vs Complex Model")
plt.xlabel("X")
plt.ylabel("Y")
plt.show()

print(f"Simple Model R²: {r2_score(y, simple_predictions):.4f}")
print(f"Complex Model R²: {r2_score(y, complex_predictions):.4f}")
print("The complex model has higher R² but is memorizing noise!")
```
</details>

## Reflection

Questions:
1. Which model looks overfit?
2. Why is memorisation dangerous?
3. Why can simpler models generalise better?



---

# Experiment 10: The Illusion of In-Sample Metrics (R² vs CV)

### Goal

Adding more variables always increases in-sample R², even if the variables are pure noise. Cross-Validation reveals the truth.

<details>
<summary>Click to view solution</summary>

```python
np.random.seed(42)
n_samples = 200

# 2 True predictive features
X_true = np.random.normal(0, 1, (n_samples, 2))
y = 5 * X_true[:, 0] - 3 * X_true[:, 1] + np.random.normal(0, 1, n_samples)

# 20 Pure Noise features
X_noise = np.random.normal(0, 1, (n_samples, 20))

# Combine
X_all = np.hstack([X_true, X_noise])

# Track metrics as we add features
in_sample_r2 = []
adj_r2 = []
cv_r2 = []

for i in range(1, 23):
    X_subset = X_all[:, :i]
    
    lr = LinearRegression().fit(X_subset, y)
    in_sample_r2.append(lr.score(X_subset, y))
    
    n = X_subset.shape[0]
    p = X_subset.shape[1]
    r2 = lr.score(X_subset, y)
    adj_r2.append(1 - (1 - r2) * (n - 1) / (n - p - 1))
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(lr, X_subset, y, cv=kf, scoring='r2')
    cv_r2.append(cv_scores.mean())

# Plot
plt.figure(figsize=(10, 6))
features_added = range(1, 23)

plt.plot(features_added, in_sample_r2, 'o-', label='In-Sample R² (Always goes up!)', color='red')
plt.plot(features_added, adj_r2, 's--', label='Adjusted R² (Penalizes noise)', color='orange')
plt.plot(features_added, cv_r2, '^-', label='5-Fold CV R² (The Truth)', color='green')

plt.axvline(2, color='gray', linestyle=':', label='True Features End (x=2)')
plt.xlabel('Number of Features Added')
plt.ylabel('R² Score')
plt.title('The Illusion of In-Sample Metrics: Adding Pure Noise')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('output/r2_vs_cv_overfitting.png', dpi=300)
plt.show()
```
</details>

### 🤔 Reflection Prompts
1. The red line (In-Sample R²) continues to climb even after feature 2, despite features 3-22 being pure noise. Why does OLS do this?
2. The green line (CV R²) peaks at 2 features and then drops. Why does adding noise features hurt out-of-sample performance?
3. **ML Relevance**: If doing automated feature selection (e.g., `SelectKBest`), why wrap the selection inside the CV loop?



---

# Experiment 11: Residual Behaviour

## Question

How do residuals behave in good vs bad models?

### Hypothesis

Good models produce random residuals with no patterns.

<details>
<summary>Click to view solution</summary>

```python
x = np.random.normal(50, 10, 300)
y = x * 5 + np.random.normal(0, 20, 300)

x_reshaped = x.reshape(-1, 1)
model = LinearRegression().fit(x_reshaped, y)
predictions = model.predict(x_reshaped)
residuals = y - predictions

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(predictions, residuals, alpha=0.5)
axes[0].axhline(0, linestyle="--", color='red')
axes[0].set_xlabel("Predicted Values")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residual Plot (Check for patterns)")

axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel("Residuals")
axes[1].set_title("Residual Distribution")

plt.tight_layout()
plt.show()
```
</details>

## Reflection

Questions:
1. Do residuals look random?
2. Why should patterns worry us?
3. What do residual patterns suggest about model adequacy?


---

# Experiment 12: Feature Importance Intuition

## Question

Which features matter most in a regression model?

### Hypothesis

Useful variables should have larger coefficients.

<details>
<summary>Click to view solution</summary>

```python
dataset = pd.DataFrame({
    "Feature_A": np.random.normal(50, 10, 500),
    "Feature_B": np.random.normal(50, 10, 500),
    "Feature_C": np.random.normal(50, 10, 500)
})

dataset["Target"] = (
    dataset["Feature_A"] * 10 +
    dataset["Feature_B"] * 2 +
    np.random.normal(0, 20, 500)
)

X = dataset.drop("Target", axis=1)
y = dataset["Target"]
model = LinearRegression().fit(X, y)

importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
}).sort_values(by="Coefficient", ascending=False)

print(importance)
print("\nFeature_A has the largest coefficient (true effect = 10)")
print("Feature_B has a smaller coefficient (true effect = 2)")
print("Feature_C has near-zero coefficient (no true effect)")
```
</details>

## Reflection

Questions:
1. Which feature mattered most?
2. Why can coefficient size mislead (e.g., with different scales)?
3. Can correlation distort importance?



---

# Experiment 13: The Danger of Extrapolation & Runge's Phenomenon

### Goal

Compare Linear, Polynomial, and Spline regressions when forced to predict outside their training domain.

<details>
<summary>Click to view solution</summary>

```python
# Generate non-linear data (logarithmic curve)
np.random.seed(42)
x_train = np.random.uniform(1, 5, 100)
y_train = 10 * np.log(x_train) + np.random.normal(0, 0.5, 100)

# Test on wider range
x_test = np.linspace(1, 8, 200)
y_true_test = 10 * np.log(x_test)

# 1. Linear Regression
lr = LinearRegression().fit(x_train.reshape(-1, 1), y_train)
y_pred_lr = lr.predict(x_test.reshape(-1, 1))

# 2. Polynomial Regression (Degree 5)
poly = PolynomialFeatures(degree=5)
x_train_poly = poly.fit_transform(x_train.reshape(-1, 1))
x_test_poly = poly.transform(x_test.reshape(-1, 1))
pr = LinearRegression().fit(x_train_poly, y_train)
y_pred_pr = pr.predict(x_test_poly)

# 3. Cubic Spline
df_train = pd.DataFrame({'x': x_train, 'y': y_train})
spline_model = smf.ols('y ~ bs(x, df=4, degree=3)', data=df_train).fit()
df_test = pd.DataFrame({'x': x_test})
y_pred_spline = spline_model.predict(df_test)

# Visualization
plt.figure(figsize=(10, 6))
plt.scatter(x_train, y_train, color='black', alpha=0.6, label='Training Data', zorder=5)
plt.plot(x_test, y_true_test, color='gray', linestyle=':', linewidth=2, label='True Function (Log)')
plt.plot(x_test, y_pred_lr, color='blue', label='Linear')
plt.plot(x_test, y_pred_pr, color='red', label='Polynomial Deg 5 (Explodes!)')
plt.plot(x_test, y_pred_spline, color='green', label='Cubic Spline')

plt.axvspan(1, 5, color='yellow', alpha=0.2, label='Training Domain')
plt.axvline(5, color='orange', linestyle='--')
plt.text(5.1, 16, 'Extrapolation\nZone', color='orange', fontsize=10)
plt.xlabel('X')
plt.ylabel('Y')
plt.title('The Danger of Extrapolation: Linear vs Polynomial vs Spline')
plt.legend(loc='lower right')
plt.ylim(-5, 25)
plt.tight_layout()
plt.savefig('output/extrapolation_danger.png', dpi=300)
plt.show()
```
</details>

### 🤔 Reflection Prompts
1. Notice how the Polynomial model completely explodes past x=6. Why does adding more degrees make extrapolation *worse*?
2. How do Tree-based models (Random Forests) handle extrapolation compared to these models?
3. **ML Relevance**: How would you implement "guardrails" in an API to prevent extrapolation errors?


---

# Experiment 14: Outliers, Leverage, and Robust Regression

### Goal

OLS minimizes squared residuals, so a single massive outlier pulls the entire regression line. Robust Regression (Huber Loss) fixes this.

<details>
<summary>Click to view solution</summary>

```python
# Generate clean linear data
np.random.seed(42)
x_clean = np.random.uniform(0, 10, 50)
y_clean = 3 * x_clean + 5 + np.random.normal(0, 2, 50)

# Inject ONE high-leverage outlier
x_outlier = np.append(x_clean, 12)
y_outlier = np.append(y_clean, -10)  # Massive negative residual at far right

# 1. Standard OLS
ols = LinearRegression().fit(x_outlier.reshape(-1, 1), y_outlier)
y_pred_ols = ols.predict(x_outlier.reshape(-1, 1))

# 2. Robust Regression (Huber)
huber = HuberRegressor().fit(x_outlier.reshape(-1, 1), y_outlier)
y_pred_huber = huber.predict(x_outlier.reshape(-1, 1))

# True line
y_true = 3 * x_outlier + 5

# Visualization
plt.figure(figsize=(10, 6))
plt.scatter(x_clean, y_clean, color='blue', alpha=0.6, label='Clean Data')
plt.scatter(12, -10, color='red', marker='X', s=150, label='High-Leverage Outlier', zorder=5)
plt.plot(x_outlier, y_true, color='gray', linestyle=':', linewidth=2, label='True Line (y=3x+5)')
plt.plot(x_outlier, y_pred_ols, color='orange', linewidth=2, label='OLS (Pulled by outlier)')
plt.plot(x_outlier, y_pred_huber, color='green', linewidth=2, linestyle='--', label='Huber Robust Regression')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('OLS vs Robust Regression: The Magnet Effect of High Leverage')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/robust_regression_outlier.png', dpi=300)
plt.show()

print(f"OLS Coefficient: {ols.coef_[0]:.3f} (True: 3.0)")
print(f"Huber Coefficient: {huber.coef_[0]:.3f} (True: 3.0)")
```
</details>

### 🤔 Reflection Prompts
1. Notice how OLS tilts down to accommodate the single red 'X'. How does this affect predictions for the bulk of data?
2. **ML Relevance**: How could you use `HuberRegressor` or `RANSAC` in an automated ML pipeline to prevent data-entry errors from corrupting daily model retraining?


---

# ML Relevance

## Why Chapter 4 matters in ML

Regression teaches prediction thinking:

### Feature Importance
Which variables matter for prediction?

### Overfitting
Memorisation is bad; generalisation is good.

### Residual Thinking
Always ask: Where does the model fail?

### Encoding
Models require numbers, not text.

### Extrapolation
Never trust models outside training domain.

### Robustness
OLS breaks with outliers; robust methods are essential in production.


---

## 📝 Experiment Log

| Date | Experiment | Key Insight | ML Relevance |
|------|-----------|-------------|--------------|
| | Extrapolation | Polynomials explode outside training bounds (Runge's Phenomenon) | Must implement API guardrails or use Tree-based models |
| | Multicollinearity | VIF > 10 inflates standard errors, destroying p-values | Don't drop correlated features for prediction; use Ridge/Lasso |
| | Robust Regression | OLS is a "magnet" for high-leverage outliers | Essential for automated pipelines |
| | R² vs CV | In-sample R² rewards pure noise; CV exposes overfitting | Always use K-Fold CV for model selection |

*Update this table as you complete experiments.*

---

## 📊 Exporting Your Findings

<details>
<summary>Click to view solution</summary>

```python
# Example: Save the R² vs CV plot
plt.figure(figsize=(10, 6))
features_added = range(1, 23)

plt.plot(features_added, in_sample_r2, 'o-', label='In-Sample R²', color='red', linewidth=2)
plt.plot(features_added, adj_r2, 's--', label='Adjusted R²', color='orange', linewidth=2)
plt.plot(features_added, cv_r2, '^-', label='5-Fold CV R²', color='green', linewidth=2)

plt.axvline(2, color='gray', linestyle=':', label='True Features End')
plt.xlabel('Number of Features Added', fontsize=12)
plt.ylabel('R² Score', fontsize=12)
plt.title('The Illusion of In-Sample Metrics: Adding Pure Noise', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('output/r2_vs_cv_final.png', dpi=300, bbox_inches='tight')
print("Saved plot to output/r2_vs_cv_final.png")
plt.show()
```
</details>

---

# Challenge Experiments

Try building experiments for:
1. Extreme overfitting with very high-degree polynomials
2. Perfect multicollinearity (exact linear relationship between features)
3. Missing important variable (omitted variable bias)
4. Tiny dataset vs huge dataset comparison
5. Noise vs prediction accuracy across different model types
6. Compare OLS, Ridge, and Lasso on correlated features

---

# Progress Checklist

- [ ] Explored noise vs signal
- [ ] Tested prediction quality across noise levels
- [ ] Investigated coefficient estimation accuracy
- [ ] Compared simple vs multiple regression models
- [ ] Explored sample size effects on stability
- [ ] Explored multicollinearity and VIF
- [ ] Tested confounding variable effects
- [ ] Simulated overfitting with polynomials
- [ ] Compared in-sample R² vs CV R²
- [ ] Analysed residual behaviour
- [ ] Understood feature importance
- [ ] Tested extrapolation dangers
- [ ] Compared OLS vs Robust Regression
- [ ] Connected regression to ML
- [ ] Exported key plots

---

> **Pro Tip**: The `HuberRegressor` and `VIF` calculations you built here are highly reusable. Save them in your `utils/stats_helpers.py` file. In the real world, running a quick VIF check on your final feature set before handing a model over to stakeholders is a hallmark of a mature Data Scientist who cares about model stability and interpretability.
